# Assignment 5 — Interactive Visualizations: Diamond Weights and Baby Names

**Intro to Data Visualization (Fall 2026)**

Group members: _<add names and student numbers>_

Run every cell from top to bottom (*Run All*). Each figure is redrawn whenever a widget changes.
Only `pandas`, `matplotlib`, `seaborn` and `ipywidgets` are imported.

In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

## Question 1 — Interactive heat map: average diamond weight by clarity and colour

Each cell shows the **mean carat** of the diamonds with that colour and clarity.
The *Origin* dropdown filters the data to natural diamonds, lab-grown diamonds, or all of them.

* Rows run from colourless **D** (top) to yellow-ish **J** (bottom), and columns from **SI2** (most included) to **IF** (internally flawless). That puts colourless, flawless diamonds in the top-right corner.
* The colour scale is fixed to 0.5–1.4 carats with the `mako` colour map, so colours mean the same weight for every filter.
* A combination with no diamonds is shown as a grey hatched cell rather than a colour. The only one is lab-grown J / IF.

In [2]:
diamonds = pd.read_csv("diamonds.csv")

COLOUR_ORDER = ["D", "E", "F", "G", "H", "I", "J"]             # colourless -> yellow-ish
CLARITY_ORDER = ["SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]  # most included -> flawless
ORIGIN_FILTERS = {"Natural": "natural", "Lab": "lab", "All": None}   # dropdown label -> value in `type`


def mean_carat_table(origin):
    """Mean carat for each colour (rows) and clarity (columns) for the chosen origin."""
    diamond_type = ORIGIN_FILTERS[origin]
    data = diamonds if diamond_type is None else diamonds[diamonds["type"] == diamond_type]
    table = data.pivot_table(index="colour", columns="clarity", values="carat", aggfunc="mean")
    return table.reindex(index=COLOUR_ORDER, columns=CLARITY_ORDER)


def draw_heatmap(origin):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.patch.set_hatch("///")          # empty (no data) cells show this hatched background
    ax.patch.set_edgecolor("#B0B0B0")
    sns.heatmap(
        mean_carat_table(origin),
        cmap="mako",
        vmin=0.5,
        vmax=1.4,
        square=True,
        ax=ax,
        cbar_kws={"ticks": [t / 10 for t in range(5, 15)], "label": "Average carat"},
    )
    ax.set_xlabel("Clarity")
    ax.set_ylabel("Colour")
    plt.show()


origin_dropdown = widgets.Dropdown(options=list(ORIGIN_FILTERS), value="Natural", description="Origin")
heatmap_output = widgets.interactive_output(draw_heatmap, {"origin": origin_dropdown})
widgets.VBox([origin_dropdown, heatmap_output])

## Question 2 — Filtering experience: top baby names

A recreation of the ONS *Top baby names* chart. It shows how the **top 100 names of 2020** ranked each year from 1996 to 2020.

* **Toggle buttons** switch between baby boy names and baby girl names.
* **Name box:** type one or more names separated by spaces. Matching ignores case, so `otis`, `OTIS` and `Otis` all work. Each name found in the 2020 top 100 is coloured and gets a point for every year.
* At most **6** names are highlighted, in the order they were typed, using the six ONS colours. The legend writes each name with a capital first letter.
* A note under the name box lists names that are not in the 2020 top 100 and names beyond the sixth. That way an unmatched name is reported instead of silently ignored.
* A name that fell outside the top 1000 in some years has a **gap** in its line for those years, since there is no rank to plot.

In [3]:
names = pd.read_csv("names.csv")

SEX_OPTIONS = {"Baby boy names": "boy", "Baby girl names": "girl"}  # toggle label -> value in `sex`
HIGHLIGHT_COLOURS = ["#206095", "#004662", "#A8BD3A", "#27A0CC", "#118C7B", "#F66068"]
MAX_HIGHLIGHTS = len(HIGHLIGHT_COLOURS)
YEARS = range(1996, 2021)


def rank_table(sex):
    """Year x name table of ranks for the top 100 names of 2020 (NaN when a name is unranked)."""
    by_sex = names[names["sex"] == sex]
    top_2020 = by_sex.loc[(by_sex["year"] == 2020) & (by_sex["rank"] <= 100), "name"]
    ranks = by_sex[by_sex["name"].isin(top_2020)].pivot(index="year", columns="name", values="rank")
    return ranks.reindex(YEARS)


# The tables do not depend on the widgets, so build them once instead of on every keystroke.
RANKS = {sex: rank_table(sex) for sex in SEX_OPTIONS.values()}


def parse_names(text, available):
    """Split the text box into (highlighted, over_limit, not_found), keeping typing order."""
    requested = list(dict.fromkeys(text.upper().split()))  # case-insensitive, duplicates removed
    found = [name for name in requested if name in available]
    not_found = [name for name in requested if name not in available]
    return found[:MAX_HIGHLIGHTS], found[MAX_HIGHLIGHTS:], not_found


def status_message(sex_label, over_limit, not_found):
    notes = []
    if not_found:
        listed = ", ".join(name.capitalize() for name in not_found)
        notes.append(f"Not in the 2020 top 100 {sex_label.lower()}: {listed}")
    if over_limit:
        listed = ", ".join(name.capitalize() for name in over_limit)
        notes.append(f"Only {MAX_HIGHLIGHTS} names can be highlighted; not shown: {listed}")
    return "<br>".join(f"<span style='color:#707070'>{note}</span>" for note in notes)


def draw_names(sex_label, text):
    ranks = RANKS[SEX_OPTIONS[sex_label]]
    highlighted, over_limit, not_found = parse_names(text, ranks.columns)
    status.value = status_message(sex_label, over_limit, not_found)

    fig, ax = plt.subplots(figsize=(10, 5.5))

    # Context: every top-100 name as a faded grey line (one call draws all 100 columns).
    ax.plot(ranks.index, ranks.to_numpy(), color="#A0A0A0", alpha=0.3, linewidth=1, zorder=1)

    # Focus: the requested names, in typing order, each with a point at every year.
    # The line carries the legend label; the points are drawn separately so the legend
    # shows a plain line, as in the original.
    for colour, name in zip(HIGHLIGHT_COLOURS, highlighted):
        ax.plot(ranks.index, ranks[name], color=colour, linewidth=2, label=name.capitalize(), zorder=3)
        ax.plot(ranks.index, ranks[name], color=colour, linestyle="none", marker="o", markersize=4, zorder=3)

    ax.set_title("Popularity ranking (1 being the most popular)", fontsize=14)
    ax.set_ylim(-15, 1050)
    ax.invert_yaxis()                                     # rank 1 at the top
    ax.set_yticks([1] + list(range(100, 1001, 100)))
    ax.set_xticks(range(1996, 2021, 2))
    ax.set_xlim(1995.5, 2020.5)
    ax.grid(axis="y", color="#D9D9D9", linewidth=1)       # horizontal lines at the y ticks
    ax.set_axisbelow(True)
    for side in ("left", "right", "top"):
        ax.spines[side].set_visible(False)
    if highlighted:
        ax.legend(loc="lower right", frameon=True)
    plt.show()


sex_toggle = widgets.ToggleButtons(options=list(SEX_OPTIONS), value="Baby boy names",
                                   style={"button_width": "150px"})
name_box = widgets.Text(value="otis mohammad charlie bobby", description="Name:",
                        placeholder="Type names separated by spaces",
                        layout=widgets.Layout(width="320px"))
status = widgets.HTML()
names_output = widgets.interactive_output(draw_names, {"sex_label": sex_toggle, "text": name_box})
widgets.VBox([sex_toggle, name_box, status, names_output])

## Design decisions (summary)

**Question 1**
* A diverging or rainbow map would suggest a meaningful midpoint, so a *sequential* map (`mako`) is used for mean carat. Weight only goes from lighter to heavier.
* The colour bar is fixed to 0.5–1.4 rather than rescaled for each filter. The same colour then means the same weight whether *Natural*, *Lab* or *All* is selected, so the views can be compared. Lab means above 1.4 (D/E-IF) show at the top colour.
* Both categorical axes follow the grading scale rather than alphabetical order. The top-right corner is the best grade (D, IF) and the bottom-left the worst (J, SI2).
* The table is rebuilt from the full data each time the dropdown changes, so the figure always matches the selection. A missing combination is hatched instead of being filled with a made-up value. Plain white would look almost the same as the palest colour of `mako`.

**Question 2**
* All 100 names are drawn in faded grey as context, and up to six chosen names are drawn in strong colours on top. That keeps the overall pattern visible while bringing the requested names forward.
* The rank tables for boys and girls are computed once. Each change of the toggle or text box then only re-plots, which keeps typing responsive.
* Input is upper-cased and de-duplicated so matching ignores case. Colours are assigned in the order names are typed, so a name keeps its colour while more names are added after it.
* The y-axis is inverted so a better rank sits higher, matching how people read "rising" popularity. Ticks at 1, 100, …, 1000 with horizontal grid lines make ranks easy to read off.
* Removing the left, right and top borders and the axis labels follows the ONS style. The title already explains the scale and the years are self-explanatory.